## VECTOR STORES AND RETRIEVERS

- Langchain vector store and retriever abstractions
- These abstractions are designed to suppoert retrieval of data -- from vector databases and other sources -- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval - augmented generation.

- Documents
- Vector Stores
- Retrievers

### DOCUMENTS

Langchain implements a document abstraction, which is intended to represent a unit of text and associated metadata.
It has two attributes.

- Pagecontent : a string representing the content
- metadata : a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companians, known for their loyalty and affection.",
        metadata={"source": "https://en.wikipedia.org/wiki/Dog"},
    ),
    Document(
        page_content="Cats are independent animals, often valued for their companionship and hunting skills.",
        metadata={"source": "https://en.wikipedia.org/wiki/Cat"},
    ),
    Document(
        page_content="Birds are a group of warm-blooded vertebrates constituting the",
        metadata={"source": "https://en.wikipedia.org/wiki/Bird"},
    )
]

documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Bird'}, page_content='Birds are a group of warm-blooded vertebrates constituting the')]

In [3]:
## vector store

from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")

e:\GitHub\Python\Python_Agentic_AI_With_Langchain_and_Langraph\practice_agentic_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### all-MiniLM-L6-v2
This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_token = os.getenv("HF_TOKEN")
HF_TOKEN = hf_token

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 453.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
### VectorStores

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embedding=embedding,
)
vectorstore

In [6]:
vectorstore.similarity_search("Cat")

[Document(id='3517df8c-fe99-4477-bc09-402f965ef0d5', metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.'),
 Document(id='17a078f8-3943-4330-abad-7558a42c0feb', metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.'),
 Document(id='51767ae9-aa4e-41a0-ae3e-d2918d8a71c4', metadata={'source': 'https://en.wikipedia.org/wiki/Bird'}, page_content='Birds are a group of warm-blooded vertebrates constituting the')]

In [7]:
## Async query

await vectorstore.asimilarity_search("Cat")

[Document(id='3517df8c-fe99-4477-bc09-402f965ef0d5', metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.'),
 Document(id='17a078f8-3943-4330-abad-7558a42c0feb', metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.'),
 Document(id='51767ae9-aa4e-41a0-ae3e-d2918d8a71c4', metadata={'source': 'https://en.wikipedia.org/wiki/Bird'}, page_content='Birds are a group of warm-blooded vertebrates constituting the')]

In [8]:
vectorstore.similarity_search_with_score("Cat")

[(Document(id='3517df8c-fe99-4477-bc09-402f965ef0d5', metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.'),
  1.0063375234603882),
 (Document(id='17a078f8-3943-4330-abad-7558a42c0feb', metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.'),
  1.5682233572006226),
 (Document(id='51767ae9-aa4e-41a0-ae3e-d2918d8a71c4', metadata={'source': 'https://en.wikipedia.org/wiki/Bird'}, page_content='Birds are a group of warm-blooded vertebrates constituting the'),
  1.647787094116211)]

### Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievals are runnables, so they implement a standerd set of methods (eg. synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chain.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [11]:
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever =  RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["Cat","Dog"])



[[Document(id='3517df8c-fe99-4477-bc09-402f965ef0d5', metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.')],
 [Document(id='17a078f8-3943-4330-abad-7558a42c0feb', metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.')]]

Vectorestores implement an as_retriever method that will generate a Retriever, specifically a vectorStoreRetriever. These retrievers include spicific search type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the folllowing -

In [12]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)
retriever.batch(["Cat","Dog"])

[[Document(id='3517df8c-fe99-4477-bc09-402f965ef0d5', metadata={'source': 'https://en.wikipedia.org/wiki/Cat'}, page_content='Cats are independent animals, often valued for their companionship and hunting skills.')],
 [Document(id='17a078f8-3943-4330-abad-7558a42c0feb', metadata={'source': 'https://en.wikipedia.org/wiki/Dog'}, page_content='Dogs are great companians, known for their loyalty and affection.')]]

In [ ]:
## Retriever along with the chain
## This is Basic RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question using the provided context only.

{question}

context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

input ="tell me about cats"
response = rag_chain.invoke(input)
print(response.content)

Cats are independent animals, often valued for their companionship and hunting skills.
